In [25]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [26]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [ ]:

queryAddress="""
SELECT 
    [City], 
    [StateProvinceID], 
    [PostalCode]
FROM Person.Address
"""
tablaAddress = pd.read_sql_query(queryAddress, motorBaseDatos)




queryStateProvince= """
SELECT 
    [StateProvinceID],
    [StateProvinceCode],
    [CountryRegionCode],
    [Name],
    [TerritoryID]
FROM Person.StateProvince
"""
tablaStateProvince = pd.read_sql_query(queryStateProvince, motorBaseDatos)

queryCountryRegion= """
SELECT 
[CountryRegionCode]
    ,[Name]
FROM Person.CountryRegion
"""
tablaCountryRegion = pd.read_sql_query(queryCountryRegion, motorBaseDatos)




# tablaSalesTerritory
# tablaAddress
# tablaStateProvince
# tablaCountryRegion



TRANSFORMACION

In [28]:

dimensionGeography = tablaAddress.merge(tablaStateProvince, on='StateProvinceID')
dimensionGeography.rename(columns=
    {
        'Name' : 'StateProvinceName'
    }, inplace=True)

dimensionGeography.drop(columns=
{
    'StateProvinceID'
}, inplace=True)

    

dimensionGeography

,City,PostalCode,StateProvinceCode,CountryRegionCode,StateProvinceName,TerritoryID
0,Ottawa,K4B 1S2,ON,CA,Ontario,6
1,Burnaby,V5A 4X1,BC,CA,British Columbia,6
2,Dunkerque,59140,59,FR,Nord,7
3,Verrieres Le Buisson,91370,91,FR,Essonne,7
4,Verrieres Le Buisson,91370,91,FR,Essonne,7
...,...,...,...,...,...,...
19609,Berlin,14129,HE,DE,Hessen,8
19610,Neunkirchen,66578,SL,DE,Saarland,8
19611,Paderborn,33041,HH,DE,Hamburg,8
19612,Berlin,10791,HH,DE,Hamburg,8


In [ ]:
dimensionGeography = dimensionGeography.merge(tablaCountryRegion, on='CountryRegionCode')
dimensionGeography.rename(columns=
    {
        'Name' : 'EnglishCountryRegionName'
    }, inplace=True)
dimensionGeography

,City,PostalCode,StateProvinceCode,CountryRegionCode,StateProvinceName,TerritoryID,EnglishCountryRegionName
0,Ottawa,K4B 1S2,ON,CA,Ontario,6,Canada
1,Burnaby,V5A 4X1,BC,CA,British Columbia,6,Canada
2,Dunkerque,59140,59,FR,Nord,7,France
3,Verrieres Le Buisson,91370,91,FR,Essonne,7,France
4,Verrieres Le Buisson,91370,91,FR,Essonne,7,France
...,...,...,...,...,...,...,...
19609,Berlin,14129,HE,DE,Hessen,8,Germany
19610,Neunkirchen,66578,SL,DE,Saarland,8,Germany
19611,Paderborn,33041,HH,DE,Hamburg,8,Germany
19612,Berlin,10791,HH,DE,Hamburg,8,Germany


In [30]:

dimensionGeography["SpanishCountryRegionName"] = None
dimensionGeography["FrenchCountryRegionName"] = None
dimensionGeography["IpAddressLocator"] = None

dimensionGeography

,City,PostalCode,StateProvinceCode,CountryRegionCode,StateProvinceName,TerritoryID,EnglishCountryRegionName,SpanishCountryRegionName,FrenchCountryRegionName,IpAddressLocator
0,Ottawa,K4B 1S2,ON,CA,Ontario,6,Canada,None,None,None
1,Burnaby,V5A 4X1,BC,CA,British Columbia,6,Canada,None,None,None
2,Dunkerque,59140,59,FR,Nord,7,France,None,None,None
3,Verrieres Le Buisson,91370,91,FR,Essonne,7,France,None,None,None
4,Verrieres Le Buisson,91370,91,FR,Essonne,7,France,None,None,None
...,...,...,...,...,...,...,...,...,...,...
19609,Berlin,14129,HE,DE,Hessen,8,Germany,None,None,None
19610,Neunkirchen,66578,SL,DE,Saarland,8,Germany,None,None,None
19611,Paderborn,33041,HH,DE,Hamburg,8,Germany,None,None,None
19612,Berlin,10791,HH,DE,Hamburg,8,Germany,None,None,None


CAMBIO DE NOMBRE

In [31]:
dimensionGeography.rename(columns=
    {
        'TerritoryID' : 'SalesTerritoryKey'
    }, inplace=True)

CARGAR A LA BODEGA

In [33]:
dimensionGeography.to_sql('dimensionGeography',motorBodegaDatos, if_exists='replace',index_label='GeographyKey')

44